In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv('df_processed.csv')
camry_data = pd.read_csv('camry_processed.csv')

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = []

def evaluate_model(model_name, model, params_desc):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    rmse = np.sqrt(mse)
    results.append({
        'Модель': model_name,
        'Гиперпараметры': params_desc,
        'MAE': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2),
        'R^2': round(r2, 4)
    })
    return model



rf_1 = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
evaluate_model('Random Forest (1)', rf_1, 'n_est=100, max_depth=None')

rf_2 = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42, n_jobs=-1)
evaluate_model('Random Forest (2)', rf_2, 'n_est=100, max_depth=10, min_split=5')

rf_3 = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
evaluate_model('Random Forest (3)', rf_3, 'n_est=300, max_depth=15')
best_rf = rf_3


xgb_1 = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
evaluate_model('XGBoost (1)', xgb_1, 'n_est=100, lr=0.1, max_depth=6')

xgb_2 = XGBRegressor(n_estimators=50, learning_rate=0.3, max_depth=4, random_state=42)
evaluate_model('XGBoost (2)', xgb_2, 'n_est=50, lr=0.3, max_depth=4')

xgb_3 = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, random_state=42)
evaluate_model('XGBoost (3)', xgb_3, 'n_est=300, lr=0.05, max_depth=8')
best_xgb = xgb_3

results_df = pd.DataFrame(results)
print("\nТаблица результатов сравнения моделей:\n")
display(results_df) 


print("\nПредсказание для отдельной машины (Camry):")
camry_features = camry_data.drop(columns=['price'])
real_price = camry_data['price'].iloc[0] 
camry_pred_rf = best_rf.predict(camry_features)[0]
camry_pred_xgb = best_xgb.predict(camry_features)[0]

print(f"Прогноз Random Forest: {camry_pred_rf:,.2f}")
print(f"Прогноз XGBoost:       {camry_pred_xgb:,.2f}")
print(f"Реальная цена:         {real_price:,.2f}")


Таблица результатов сравнения моделей:



,Модель,Гиперпараметры,MAE,MSE,R^2
0,Random Forest (1),"n_est=100, max_depth=None",119354.94,3.221659e+10,0.9571
1,Random Forest (2),"n_est=100, max_depth=10, min_split=5",118197.51,3.019647e+10,0.9598
2,Random Forest (3),"n_est=300, max_depth=15",117888.51,3.099143e+10,0.9587
3,XGBoost (1),"n_est=100, lr=0.1, max_depth=6",115932.07,2.843099e+10,0.9621
4,XGBoost (2),"n_est=50, lr=0.3, max_depth=4",118988.43,2.967577e+10,0.9605
5,XGBoost (3),"n_est=300, lr=0.05, max_depth=8",114305.10,2.921302e+10,0.9611



Предсказание для отдельной машины (Camry):
Прогноз Random Forest: 1,497,330.09
Прогноз XGBoost:       1,487,572.25
Реальная цена:         1,450,000.00


![Метрики качества (Эксперимент 2)](linear_metrics.png)

![Предсказание линейной регрессии для Camry](linear_camry.png)